In [ ]:
import sklearn

In [ ]:
import numpy as np

def generate_data(n, K, sigma_UX, sigma_UY, sigma_UC, d=5, s=5):
    # Generate Pi matrix with iid N(0, 1/K) rows
    Pi = np.random.randn(K, d) / np.sqrt(K)  # (K, d)
    
    # Create a fixed s-sparse theta in R^d
    theta = np.zeros(d)
    nonzero_indices = np.random.choice(d, s, replace=False)
    theta[nonzero_indices] = 2 * (np.random.randn(s) >= 0) - 1  # Random ±1 values

    # Generate independent noise terms for each (k, i)
    U_X = np.random.randn(n, K) * sigma_UX  # (n, K)
    U_Y = np.random.randn(n, K) * sigma_UY  # (n, K)
    U_C = np.random.randn(n, K) * sigma_UC  # (n, K)

    # Compute X_{ki} ensuring dependence on U_X, U_C, and Pi
    X = Pi[None, :, :] + U_C[:, :, None] + U_X[:, :, None]  # (n, K, d)

    # Compute X_{ki}^T theta
    X_theta = np.einsum('nkd,d->nk', X, theta)  # (n, K)

    # Generate Y_{ki} using independent U_Y and U_C
    Y = X_theta + U_Y + U_C  # (n, K)

    # Reshape outputs to (n*K, d) and (n*K, )
    X_flat = X.reshape(n * K, d)
    Y_flat = Y.ravel()
    X_theta_flat = X_theta.ravel()

    # Generate one-hot encoding for category K
    one_hot_K = np.eye(K)[np.tile(np.arange(K), n)]  # (n*K, K)

    # Define h_fun function that maps any theta to X @ theta
    def h_fun(theta_new):
        return np.einsum('nkd,d->nk', X, theta_new).ravel()

    return theta, Y_flat, X_theta_flat, X_flat, one_hot_K, Pi, h_fun

# Example usage
n = 1  # Number of individuals
d = 10000
s = 5
K = int(np.ceil( 10*  s * np.log(2.71*d/s)**(2)))
print(K)
sigma_UX = 0.1
sigma_UY = 0.1
sigma_UC = 0.1

theta, Y, X_theta, X, one_hot_K, Pi, h_fun = generate_data(n, K, sigma_UX, sigma_UY, sigma_UC, d=d, s =s)
mu_Z_hat = np.array([Y[np.where(one_hot_K[:, k] == 1)].mean() for k in range(K)])

In [ ]:
import numpy as np
from sklearn.linear_model import Lasso

# Set parameters
d = 50
s = 5
K = 49  # int(np.ceil(100 * s * np.log(2.71 * d / s)))
print("K:", K)

# Generate Pi matrix
Pi = np.random.randn(K, d) / np.sqrt(K)  # (K, d)

# Create sparse theta
theta = np.zeros(d)
nonzero_indices = np.random.choice(d, s, replace=False)
theta[nonzero_indices] = 2 * (np.random.randn(s) >= 0) - 1  # Random ±1 values

# Compute expected response
mu_Z = Pi @ theta

# 1. Minimum Norm Solution
theta_hat_lstsq = np.linalg.lstsq(Pi, mu_Z, rcond=None)[0]

# 2. Lasso Solution
lasso = Lasso(alpha=0.00001, fit_intercept=False)  # Adjust alpha for optimal sparsity
lasso.fit(Pi, mu_Z)
theta_hat_lasso = lasso.coef_

# Compute errors for both methods
def compute_metrics(theta_hat, theta):
    error_proj = np.sqrt(np.mean((Pi @ (theta_hat - theta))**2))  # Error in projection
    error_rel = np.sqrt(np.sum((theta_hat - theta)**2)) / np.sqrt(np.sum(theta**2))  # Relative error
    norm_theta_hat = np.sqrt(np.sum(theta_hat**2))
    norm_theta = np.sqrt(np.sum(theta**2))
    return error_proj, error_rel, norm_theta_hat, norm_theta

# Compute metrics
metrics_lstsq = compute_metrics(theta_hat_lstsq, theta)
metrics_lasso = compute_metrics(theta_hat_lasso, theta)

# Print results
print("\nMinimum Norm Solution:")
print(f"Projection Error: {metrics_lstsq[0]:.6f}")
print(f"Relative Error: {metrics_lstsq[1]:.6f}")
print(f"Norms: Estimated = {metrics_lstsq[2]:.6f}, True = {metrics_lstsq[3]:.6f}")

print("\nLasso Solution:")
print(f"Projection Error: {metrics_lasso[0]:.6f}")
print(f"Relative Error: {metrics_lasso[1]:.6f}")
print(f"Norms: Estimated = {metrics_lasso[2]:.6f}, True = {metrics_lasso[3]:.6f}")

 

In [ ]:
import numpy as np
from sklearn.linear_model import Lasso

# Set parameters
d = 1000
s = 10
K = 50  # int(np.ceil(100 * s * np.log(2.71 * d / s)))
alpha1 = 2  # Sobolev decay parameter
alpha2 = alpha1
use_sobolev_theta = True  # Toggle between s-sparse or Sobolev-structured theta

print("K:", K)
print("Sobolev alpha:", alpha)
print("Using Sobolev theta:", use_sobolev_theta)

# Generate Pi matrix with Gaussian decay following N(0, 1/K)
decay = np.array([j**(-alpha2) for j in range(1, d + 1)])  # Sobolev-type decay
Pi = np.random.randn(K, d) * (decay[np.newaxis, :] / np.sqrt(K))  # Apply Gaussian decay

# Construct theta
theta = np.zeros(d)

if use_sobolev_theta:
    # Sobolev-structured theta: Random ±1 values multiplied by decay
    decay = np.array([j**(-alpha1) for j in range(1, d + 1)])
    theta = (2 * (np.random.rand(d) >= 0.5) - 1) * decay
else:
    # s-sparse theta: Random ±1 values in first s coordinates
    theta[:s] = 2 * (np.random.rand(s) >= 0.5) - 1

# Compute expected response
mu_Z = Pi @ theta

# 1. Minimum Norm Solution
theta_hat_lstsq = np.linalg.lstsq(Pi, mu_Z, rcond=None)[0]

# 2. Lasso Solution
lasso = Lasso(alpha=1e-10, fit_intercept=False)  # Adjust alpha for optimal sparsity
lasso.fit(Pi, mu_Z)
theta_hat_lasso = lasso.coef_

# Compute errors for both methods
def compute_metrics(theta_hat, theta):
    error_proj = np.sqrt(np.mean((Pi @ (theta_hat - theta))**2))  # Error in projection
    error_rel = np.sqrt(np.sum((theta_hat - theta)**2)) / np.sqrt(np.sum(theta**2))  # Relative error
    norm_theta_hat = np.sqrt(np.sum(theta_hat**2))
    norm_theta = np.sqrt(np.sum(theta**2))
    return error_proj, error_rel, norm_theta_hat, norm_theta

# Compute metrics
metrics_lstsq = compute_metrics(theta_hat_lstsq, theta)
metrics_lasso = compute_metrics(theta_hat_lasso, theta)

# Print results
print("\nMinimum Norm Solution:")
print(f"Projection Error: {metrics_lstsq[0]:.6f}")
print(f"Relative Error: {metrics_lstsq[1]:.6f}")
print(f"Norms: Estimated = {metrics_lstsq[2]:.6f}, True = {metrics_lstsq[3]:.6f}")

print("\nLasso Solution:")
print(f"Projection Error: {metrics_lasso[0]:.6f}")
print(f"Relative Error: {metrics_lasso[1]:.6f}")
print(f"Norms: Estimated = {metrics_lasso[2]:.6f}, True = {metrics_lasso[3]:.6f}")


# Data

In [115]:
import numpy as np

# Function to generate data
def generate_data(n, K, sigma_UX=0.1, sigma_UY=0.1, sigma_UC=0.1, d=5,  alpha1=2, alpha2=2):
    # Generate Pi matrix with Gaussian decay following N(0, 1/K)
    decay = np.array([j**(-alpha2) for j in range(1, d + 1)])  # Sobolev-type decay
    Pi = np.random.randn(K, d) * (decay[np.newaxis, :]  )  # Apply Gaussian decay

    # Construct theta
    decay = np.array([j**(-alpha1) for j in range(1, d + 1)])
    theta = (2 * (np.random.rand(d) >= 0.5) - 1) * decay  # Random ±1 values with decay

    # Generate independent noise terms for each (k, i)
    U_X = np.random.randn(n, K) * sigma_UX  # (n, K)
    U_Y = np.random.randn(n, K) * sigma_UY  # (n, K)
    U_C = np.random.randn(n, K) * sigma_UC  # (n, K)

 
    # Compute X_{ki} ensuring dependence on U_X, U_C, and Pi
    X =   Pi[None, :, :] + U_C[:, :, None] + U_X[:, :, None]  # (n, K, d)

    # Compute X_{ki}^T theta
    X_theta = np.einsum('nkd,d->nk', X, theta)  # (n, K)

    # Generate Y_{ki} using independent U_Y and U_C
    Y = X_theta + U_Y + U_C  # (n, K)

    # Reshape outputs to (n*K, d) and (n*K, )
    X_flat = X.reshape(n * K, d)
    Y_flat = Y.ravel()
    X_theta_flat = X_theta.ravel()

    # Generate one-hot encoding for category K
    Z = np.tile(np.arange(K), n)

    # Define h_fun function that maps any theta to X @ theta
    def h_fun(theta_new):
        return np.einsum('nkd,d->nk', X, theta_new).ravel()

    return theta, Y_flat, X_theta_flat, X_flat, Z, Pi, h_fun

 
def compute_group_means(data, Z, folds, fold_val, K):
    """
    Compute group means for data (which can be X or Y) for a given fold.
    data: array of shape (n_total, d) or (n_total,) for Y.
    Z: group labels (n_total,)
    folds: fold indicator array (n_total,)
    fold_val: the fold for which to compute the means (0 or 1)
    K: number of groups
    """
    idx = (folds == fold_val)
    # For multidimensional data:
    if data.ndim == 2:
        d = data.shape[1]
        group_sum = np.zeros((K, d))
        np.add.at(group_sum, Z[idx], data[idx])
    else:
        # data is 1-D (e.g. Y)
        group_sum = np.zeros(K)
        np.add.at(group_sum, Z[idx], data[idx])
    # Count number of observations per group in this fold
    counts = np.bincount(Z[idx], minlength=K).reshape(-1, 1) if data.ndim == 2 else np.bincount(Z[idx], minlength=K)
    counts[counts == 0] = 1  # Avoid division by zero
    return group_sum / counts
 
def compute_metrics(theta_hat, theta_true, Z, X, Y, folds, K):
    n_total = len(Z)
    group_counts = np.bincount(Z, minlength=K)
    W = group_counts / n_total
    res = X @ theta_true - X @ theta_hat
    res_1 = compute_group_means(res, Z, folds, 1, K)
    res_0 = compute_group_means(res, Z, folds, 0, K)
    
    strong_norm = np.sqrt(np.mean(res**2))
    weak_norm = np.sqrt(np.mean(res_1 * res_0))
    jive_risk = np.mean(W * compute_group_means(Y - X @ theta_hat, Z, folds, 1, K) * compute_group_means(Y - X @ theta_hat, Z, folds, 0, K)) / np.mean(W)
    
    print(f"{'Metric':<15}{'Value'}")
    print("=" * 30)
    print(f"{'Strong Norm':<15}{strong_norm:.6f}")
    print(f"{'Weak Norm':<15}{weak_norm:.6f}")
    print(f"{'JIVE Risk':<15}{jive_risk:.6f}")
    
    return  
 

# Example usage
n = 20 # Number of individuals
d = 300
K = 20
sigma_UX = 0.1
sigma_UY = 0.1
sigma_UC = 0.1
alpha1 = 1
alpha2 = alpha1

# Generate data
theta_true, Y, X_theta, X, Z, Pi, h_fun = generate_data(n, K, sigma_UX, sigma_UY, sigma_UC, d, alpha1, alpha2)
print(Y.shape, Z.shape)

def npJIVE(X, Z, Y, K, folds = None, lambda_K = 1e-3):
    n_total = len(Z)
    group_counts = np.bincount(Z, minlength=K)
    W = group_counts / n_total  # shape (K,)
    if folds is None:
        folds = np.zeros(n_total, dtype=int)
        folds[:n_total // 2] = 1
        np.random.shuffle(folds)

    # Compute fold means for each group (X_fold0, X_fold1: shape (K, d), Y_fold0, Y_fold1: shape (K,))
    X_fold0 = compute_group_means(X, Z, folds, 0, K)
    X_fold1 = compute_group_means(X, Z, folds, 1, K)
    Y_fold0 = compute_group_means(Y, Z, folds, 0, K)
    Y_fold1 = compute_group_means(Y, Z, folds, 1, K)

    # Compute the cross-products efficiently
    M = (X_fold0.T * W) @ X_fold1 + (X_fold1.T * W) @ X_fold0  # shape (d, d)
    b = (X_fold0.T * W) @ Y_fold1 + (X_fold1.T * W) @ Y_fold0  # shape (d,)

    S = (X.T * W[Z]) @ X    # shape (d, d), weighted sum over all observations
    # Regularized system: (M +  lambda_K * S) beta = b
    M_reg = M +   lambda_K * S
    theta_hat = np.linalg.lstsq(M_reg, b, rcond=None)[0]
    return theta_hat
    
from sklearn.model_selection import KFold

def cross_validate_lambda(X, Z, Y, K, lambda_grid, jive_folds = None, n_splits=5):
    """Perform K-fold cross-validation to select optimal lambda_K."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    best_lambda = None
    best_risk = float("inf")
    n_total = len(Z)
    group_counts = np.bincount(Z, minlength=K)
    W = group_counts / n_total  # shape (K,)
    if jive_folds is None:
        jive_folds = np.zeros(n_total, dtype=int)
        jive_folds[:n_total // 2] = 1
        np.random.shuffle(jive_folds)
    for lambda_K in lambda_grid:
        total_risk = 0
        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X[train_idx], X[val_idx]
            Y_train, Y_val = Y[train_idx], Y[val_idx]
            Z_train, Z_val = Z[train_idx], Z[val_idx]
            jive_folds_train, jive_folds_val = jive_folds[train_idx], jive_folds[val_idx]
            theta_hat = npJIVE(X_train, Z_train, Y_train, K, jive_folds_train, lambda_K)
            
            # Compute risk on validation set
            risk = np.mean(W * compute_group_means(Y_val - X_val @ theta_hat, Z_val, jive_folds_val, 1, K) * compute_group_means(Y_val - X_val @ theta_hat, Z_val, jive_folds_val, 0, K)) / np.mean(W)
            total_risk += risk
        
        avg_risk = total_risk / n_splits
        if avg_risk < best_risk:
            best_risk = avg_risk
            best_lambda = lambda_K
    
    return best_lambda, jive_folds

 
best_lambda, folds = cross_validate_lambda(X, Z, Y, K, lambda_grid = np.logspace(-5, 1, 10), n_splits=2) 
print("best_lambda: ", best_lambda)
theta_hat = npJIVE(X, Z, Y, K, folds, lambda_K = best_lambda)

  
X_all =  compute_group_means(X, Z, np.zeros_like(folds), 0, K)
Y_all =  compute_group_means(Y, Z, np.zeros_like(folds), 0, K)  
theta_hat_lstsq = np.linalg.lstsq(X_all, Y_all, rcond=None)[0]


print(Z.shape)
compute_metrics(theta_hat, theta_true, Z, X, Y, K  folds, K)
compute_metrics(theta_hat_lstsq, theta_true, Z, X, Y,   folds, K)

(400,) (400,)
best_lambda:  0.001
(400,)
Metric         Value
Strong Norm    0.078249
Weak Norm      0.032794
JIVE Risk      -0.000682
Metric         Value
Strong Norm    0.087847
Weak Norm      0.027000
JIVE Risk      -0.000743


In [121]:
def dual_solution(X, Z, K, X_new = X, folds = None, lambda_K = 1e-3):
    n_total = len(Z)
    group_counts = np.bincount(Z, minlength=K)
    W = group_counts / n_total  # shape (K,)
    if folds is None:
        folds = np.zeros(n_total, dtype=int)
        folds[:n_total // 2] = 1
        np.random.shuffle(folds)

    # Compute fold means for each group (X_fold0, X_fold1: shape (K, d), Y_fold0, Y_fold1: shape (K,))
    X_fold0 = compute_group_means(X, Z, folds, 0, K)
    X_fold1 = compute_group_means(X, Z, folds, 1, K)

    # Compute the cross-products efficiently
    M = (X_fold0.T * W) @ X_fold1 + (X_fold1.T * W) @ X_fold0  # shape (d, d)
    b = (2 / len(X_new)) * X_new.T @ np.ones(len(X_new))  # shape (d,)

    S = (X.T * W[Z]) @ X    # shape (d, d), weighted sum over all observations
    # Regularized system: (M +  lambda_K * S) beta = b
    M_reg = M +   lambda_K * S
    theta_hat = np.linalg.lstsq(M_reg, b, rcond=None)[0]
    return theta_hat


theta_hat = npJIVE(X, Z, Y, K, folds, lambda_K = 1e-3)
b_hat = dual_solution(X, Z, K,  lambda_K = 1e-3)
h_hat = X@theta_hat
beta_hat = X@b_hat
q_hat = compute_group_means(data, Z, folds, 0, K)

array([1.0381894 , 0.9774665 , 1.10728856, 0.99960295, 0.98474619,
       0.95500085, 0.99763921, 0.79171616, 1.08284341, 0.90120342,
       0.94044673, 0.97534524, 0.92802226, 0.94225051, 1.09563484,
       1.02193975, 0.96740813, 1.03248296, 0.93699851, 0.89669301,
       0.99025442, 1.15944148, 0.9536282 , 1.01457606, 1.01395213,
       0.98371405, 1.04518703, 1.11884257, 1.16257862, 0.98204986,
       1.00730467, 0.76744371, 0.77476453, 0.85450051, 0.97816142,
       0.96075424, 0.90065881, 0.95241985, 0.83062817, 0.85797702,
       1.00705409, 0.9824237 , 1.02571076, 1.18556859, 1.05229399,
       0.9152917 , 1.07257453, 1.03152698, 0.92158337, 0.94657797,
       1.04891465, 1.12946884, 1.04743172, 0.9041494 , 1.08371549,
       1.09365818, 1.17713489, 1.22863037, 0.89557933, 0.90561297,
       0.99747017, 0.9135592 , 0.86560147, 0.96713354, 1.07878081,
       0.79970791, 0.94459137, 0.97779138, 0.88704044, 1.10717939,
       0.9929768 , 1.00353915, 0.99896398, 0.9314873 , 0.98078

In [107]:
jive_risk = (T_fold1 @ (Y - X @ theta_hat)).T @ W @ T_fold0 @ (Y - X @ theta_hat) 
print(jive_risk)
jive_risk = (T_fold1 @ (Y - X @ theta_hat_lstsq)).T @ W @ T_fold0 @ (Y - X @ theta_hat_lstsq) 
print(jive_risk)

weak_norm = np.sqrt((T_fold1 @ (X @ theta_true - X @ theta_hat)).T @ W @ T_fold0 @ (X @ theta_true - X @ theta_hat) )
print(weak_norm)
weak_norm = np.sqrt((T_fold1 @ (X @ theta_true - X @ theta_hat_lstsq)).T @ W @ T_fold0 @ (X @ theta_true - X @ theta_hat_lstsq) )
print(weak_norm)


strong_norm = np.sqrt(np.mean((X @ theta_true - X @ theta_hat)**2) )
print(strong_norm)
strong_norm = np.sqrt(np.mean((X @ theta_true - X @ theta_hat_lstsq)**2) )
print(strong_norm)

NameError: name 'T_fold1' is not defined

In [ ]:
print(np.sqrt(np.sum((theta_hat - theta_true)**2)) / np.sqrt(np.sum(theta_true**2)))
print(np.sqrt(np.sum((theta_hat_lstsq - theta_true)**2)) / np.sqrt(np.sum(theta_true**2)))
 

In [ ]:
print(np.sqrt(np.mean((Pi @ theta_true - Pi @ theta_hat)**2)) / np.sqrt(np.mean((Pi @ theta_true)**2)))
print(np.sqrt(np.mean((Pi @ theta_true - Pi @ theta_hat_lstsq)**2)) / np.sqrt(np.mean((Pi @ theta_true)**2)))

print(np.sqrt(np.mean((X @ theta_true - X @ theta_hat)**2)) )
print(np.sqrt(np.mean((T_all @ X @ theta_true - T_all @ X @ theta_hat)**2))) 
